# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The notebook uses the package's default OpenAI `gpt-4.1-mini` client. Set `OPENAI_API_KEY` in the environment before running the model cell. The key is read by the OpenAI SDK and is never stored in this notebook.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=2, cache_dir=ROOT / 'data' / 'raw')
[(story.splitlines()[0], len(story)) for story in stories]

[('THE DAW IN BORROWED FEATHERS', 859), ('THE SUN AND THE WIND', 972)]

In [2]:
len(stories), [story.splitlines()[0] for story in stories]

(2, ['THE DAW IN BORROWED FEATHERS', 'THE SUN AND THE WIND'])

## OpenAI model

The transformer creates the default `OpenAIModelClient` when `model` is omitted. It uses `gpt-4.1-mini` through the Responses API and reads `OPENAI_API_KEY` from the environment.

In [ ]:
graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
)
traces = graphicalizer.fit([stories[0]]).transform_with_trace([stories[0]])
trace = traces[0]
[(p.text, p.proposition_id) for p in trace.propositions]

[SemanticGraphicalizer] ready: model=OpenAIModelClient, ontology=aesop-narrative, domain=aesop-narrative
[document-71c1b92da992] processing document: chars=859
[document-71c1b92da992] segment: 1 -> 1 | 0.1 ms | input_chars=859, chunk_chars=859
[document-71c1b92da992 chunk-0] compiling semantic stages
[document-71c1b92da992 chunk-0] summarize: 1 -> 1 | 7.4 s | input_chars=859, output_chars=488
[document-71c1b92da992 chunk-0] normalize: 1 -> 1 | 3.6 s | input_chars=488, output_chars=697
[document-71c1b92da992 chunk-0] decompose: 1 -> 13 | 9.1 s


In [ ]:
graphicalizer.display(trace.graph, mode="text")

In [ ]:
# Deterministic alternative: NetworkX Kamada-Kawai layout.
graphicalizer.display(trace.graph, mode="static")

In [ ]:
graphicalizer.display(trace.graph, mode="dynamic", charge_strength=-1)